# Day 11(M3 Day02) 실습 — PostgreSQL 세션 메모리와 사용자별 문맥 유지

> **용어 기준:** 대화 이력·사용자 정보를 활용하는 기능은 메모리(memory), 실행 중 단계 사이에 전달·갱신되는 데이터는 상태(state)다. 체크포인트(checkpoint)는 저장된 실행 상태이며, 체크포인트 저장기(checkpointer)가 이를 저장·복원한다. 서로 관련은 있지만 같은 개념은 아니다.


**목표**: 1과목 챗봇에서 Supabase 클라이언트로 했던 대화 저장·불러오기를 SQL과 LangChain으로 다시 쓰고, SQL로 직접 다룰 때 만나는 문제를 익힌다. 도구 호출(M2)과 결합해 에이전트가 기억을 스스로 꺼내고 저장하게 한다.

| Part | 내용 |
| --- | --- |
| Part 1 | 1과목 복습 — 저장·불러오기, role 변환, 세션 분리, 최근 N개를 SQL·LangChain으로 |
| Part 2 | SQL 직접 접속에서 만나는 것 — 문자열 끼워 넣기, 컬럼명 오타, 테이블 이름 충돌, 트랜잭션, 인덱스, 세션 삭제, 1과목 구조 비교 |
| Part 3 | 면접 코치 — 세션 기억 + Day01 지식 검색 |
| Part 4 | 도구 호출과 세션 메모리 — 기억 검색 도구, 사실 기억 도구 / 선택: 도구 호출 기록 저장, 짝 깨짐 오류, `create_agent` |
| 확장 (별도 노트북) | 투자 리서치 어시스턴트 — `Day11-확장-투자리서치어시스턴트-실습.ipynb` |

> **참고:** 실습 전 가상환경 활성화, `.env`의 `OPENAI_API_KEY`·`DATABASE_URL`을 확인한다. `DATABASE_URL`은 Supabase **Session pooler** 주소다(배포 문서 0절). `uv add psycopg2-binary`가 필요하다(이미 설치돼 있다면 생략).

## 0. 환경 준비 (DB 연결)

In [7]:
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGCHAIN_TRACING_V2"] = "false"

import psycopg2
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

from dotenv import load_dotenv
import numpy as np

In [8]:
load_dotenv()
conn = psycopg2.connect(os.getenv("DATABASE_URL"))
conn.autocommit = True

cur = conn.cursor()
llm = ChatOpenAI(model="gpt-4o-mini")
emb = OpenAIEmbeddings(model="text-embedding-3-small")

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

print("준비 완료")

준비 완료


In [9]:
cur.execute("select version()")
cur.fetchone()

('PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit',)

In [ ]:
cur.execute("select table_name from information_schema.tables where table_schema = 'public'") # sql문 실행
cur.fetchall() #결과추출

[('conversations',), ('messages',), ('profiles',)]

## Part 1. 1과목 복습 — SQL과 LangChain으로 다시 쓰기

1과목 챗봇(19일차)에서 대화를 저장하고, 이전 대화를 모델에 함께 보냈다. 같은 일을 SQL과 LangChain으로 다시 쓴다. 1과목에서 정리한 **기록**(테이블에 남는 것)과 **기억**(모델에 보내는 것)의 구분은 그대로다.

### 1-1. session_messages 테이블 생성

대화를 담을 표를 만든다.

> **참고:** 1과목 Supabase 프로젝트에는 챗봇용 `messages` 테이블이 이미 있다. `CREATE TABLE IF NOT EXISTS`는 같은 이름의 테이블이 있으면 오류 없이 건너뛰기 때문에, 이름을 `messages`로 쓰면 1과목 테이블에 저장하게 되어 `column "session_id" ... does not exist` 오류가 난다. 그래서 `session_messages`라는 이름을 따로 쓴다.

In [ ]:
# 대화 id > session_id

cur.execute("""
CREATE TABLE IF NOT EXISTS session_messages (
    id serial PRIMARY KEY,
    session_id text,
    role text,
    content text,
    created_at timestamptz DEFAULT now()
)""")
# 실습을 처음부터 다시 돌릴 때 이전 실행의 메시지가 섞이지 않도록 비운다
cur.execute("TRUNCATE session_messages RESTART IDENTITY")
print("테이블 준비 완료")

테이블 준비 완료


### 1-2. 1과목 코드와 대응 — 저장·불러오기·role 변환

| 하는 일 | 1과목 (Supabase 클라이언트, Gemini) | 오늘 (SQL, LangChain) |
| --- | --- | --- |
| 메시지 저장 | `create_message()` | `save()` — `INSERT` |
| 대화 불러오기 | `list_messages()` | `load()` — `SELECT ... ORDER BY id` |
| role 변환 | `_ROLE_MAP = {"user": "user", "assistant": "model"}` | `ROLE_TO_MESSAGE` — 메시지 클래스로 |
| 최근 N개 | `usable[-MAX_HISTORY_MESSAGES:]` | `load_recent()` — `ORDER BY id DESC LIMIT` |
| 대화 한 턴 | `_build_history()` + 모델 호출 | `chat()` |

`%s`는 값을 넣을 자리표시자다. 값을 SQL 문자열에 직접 이어 붙이지 않는다(Part 2 2-1에서 이유를 확인한다).

In [11]:
def save(session, role, content):
    """session_messages 테이블에 메시지 추가합니다."""
    cur.execute(
        """
        INSERT INTO session_messages (session_id, role, content) 
        VALUES (%s, %s, %s)
        """,
        (session, role, content),
    )


def load(session):
    """특정 대화(세션id)를 가져옵니다."""
    cur.execute(
        """
        SELECT role, content FROM session_messages WHERE session_id = %s 
        ORDER BY id
        """,
        (session,),
    )
    return cur.fetchall()


def load_recent(session, n):
    """특정 대화의 최근 n개 메시지를 대화 순서로 가져옵니다."""
    cur.execute(
        """
        SELECT role, content FROM session_messages 
        WHERE session_id = %s 
        ORDER BY id DESC 
        LIMIT %s
        """,
        (session, n),
    )
    return list(reversed(cur.fetchall()))

print("함수 준비 완료")

함수 준비 완료


In [10]:
save("userA", "user", "안녕하세요")
save("userA", "assistant", "안녕하세요! 무엇을 도와드릴까요?")

In [11]:
print(load("userA"))
print(load_recent("userA", 1))

[('user', '안녕하세요'), ('assistant', '안녕하세요! 무엇을 도와드릴까요?')]
[('assistant', '안녕하세요! 무엇을 도와드릴까요?')]


In [16]:
SYSTEM_PROMPT = SystemMessage("한문장으로 짧게 답한다.")

In [17]:
ROLE_TO_MESSAGE = {"user": HumanMessage, "ai": AIMessage, "assistant": AIMessage, "system": SystemMessage}

#세션id의 모든 메시지를 조회, 메시지내용을 적절한 메시지 객체로 생성 > 리스트 누적 > 메시지 목록을 리스트로 반환
def to_messages(rows):
    out = []
    for role, content in rows:
        if role not in ROLE_TO_MESSAGE:
            raise ValueError(f"알 수 없는 role: {role!r}")
        out.append(ROLE_TO_MESSAGE[role](content))
    return out


In [18]:
def chat(session_id, message):
    save(session_id, "user", message)
    result = llm.invoke([SYSTEM_PROMPT] + to_messages(load(session_id)))
    save(session_id, "ai", result.content)
    return result.content

In [19]:
# 테스트 과정에서 생긴 특정 세셔의 모든 메시지를 지우는 함수 
def clear(session):
    cur.execute("DELETE FROM session_messages WHERE session_id=%s", (session,))

In [22]:
clear("userA")

### 1-3. 세션 분리와 최근 N개 확인

1과목에서 확인한 두 가지를 다시 본다. 세션이 다르면 기억이 섞이지 않는다. 최근 N개만 보내면 입력 토큰은 줄지만 그 밖의 대화는 잊는다.

`long` 세션에는 12턴 대화를 넣는다. AI 답은 LLM을 부르지 않도록 고정 문장으로 넣는다. 1턴에 이름, 2턴에 사는 곳을 말한다. 이 세션은 Part 4에서도 쓴다.

In [23]:
chat("userA", "내 이름은 홍길동이야")

'반갑습니다, 홍길동님!'

In [24]:
turns = [
    "내 이름은 민수야",
    "나는 서울에 살아",
    "직업은 AI Agent 기획자야",
    "부자가 되고 싶다.",
    "가을 여행지 하나 추천해줘"
]

# TODO: 최근 6개(load_recent)만 넣어 같은 질문을 하고 결과를 recent에 담으세요
# 세션은 userB로 생성
for t in turns:
    chat("userB", t)


In [26]:
print(load_recent("userB", 6))

[('user', '직업은 AI Agent 기획자야'), ('ai', '흥미로운 직업이네요! 어떤 프로젝트를 진행 중이신가요?'), ('user', '부자가 되고 싶다.'), ('ai', '부를 이루기 위해 목표를 세우고 꾸준히 노력하세요!'), ('user', '가을 여행지 하나 추천해줘'), ('ai', '가을에는 단풍이 아름다운 설악산 추천합니다!')]


### 관찰

| 확인 | 결과 | 1과목 |
| --- | --- | --- |
| 세션 분리 | userB는 userA의 이름을 모른다 | `conversation_id`와 RLS로 분리 |
| 최근 N개 | 입력 토큰이 줄지만 1턴의 이름을 잊는다 | `MAX_HISTORY_MESSAGES = 20` |

> **참고:** 오래된 대화를 요약해 넣는 방법은 M4 Day15에서 다룬다. 최근 N개 밖의 정보를 필요할 때 꺼내는 방법은 Part 4에서 다룬다.

## Part 2. SQL 직접 접속에서 만나는 것

1과목은 Supabase 클라이언트로 접속해 SQL을 직접 쓰지 않았다. 이 과목은 `DATABASE_URL`로 PostgreSQL에 직접 붙는다. 이때 처음 만나는 오류와 개념을 다룬다.

### 2-1. 오류 대응 ① — 값을 문자열에 직접 끼워 넣으면?

`%s` 자리표시자 대신 값을 SQL 문자열에 직접 이어 붙이면 어떤 일이 생기는지 확인한다.

In [ ]:
unsafe_content = ""   # 아포스트로피 포함
try:
    # TODO: unsafe_content를 SQL 문자열에 f-string으로 직접 끼워 넣어 INSERT를 실행하세요
    #       (VALUES ('demo2', 'user', '{unsafe_content}') 형태)


    print("문자열 직접 삽입 성공(위험한 코드지만 이번엔 통과)")
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", e)

# %s 자리표시자를 쓰는 안전한 방식은 같은 값도 문제 없이 저장한다

print("자리표시자 방식 저장 완료")

> **주의:** 결과는 실제 실행에서 직접 확인한다. 값을 문자열에 직접 끼워 넣으면 값 안의 따옴표 하나가 SQL 구문 자체를 깨뜨릴 수 있다 — 이것이 SQL 인젝션이 가능해지는 근본 원인이다. `%s` 자리표시자는 값과 SQL 구문을 분리해서 이런 문제 자체가 생기지 않는다.

### 2-2. 오류 대응 ② — 컬럼명을 잘못 쓰면?

SELECT에서 존재하지 않는 컬럼명을 쓰면 어떻게 되는지, 그리고 그 오류 이후에도 정상 조회가 되는지 확인한다.

In [ ]:
try:
    # TODO: 존재하지 않는 컬럼 rolee로 SELECT를 실행하세요



    print("조회 결과:", cur.fetchall())
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", e)

# 오류 이후에도 정상 조회가 되는지 확인


> **참고:** 컬럼명 오타는 `psycopg2.errors.UndefinedColumn` 오류로 즉시 드러난다. `conn.autocommit = True`인 경우, 오류가 난 다음 쿼리도 트랜잭션 롤백 없이 곧바로 이어서 실행할 수 있는지를 실제 실행으로 확인한다.

### 2-3. 오류 대응 ③ — 같은 이름의 테이블이 이미 있으면?

1-1에서 `session_messages`라는 이름을 따로 쓴 이유를 직접 재현한다. 1과목 챗봇 테이블처럼 `conversation_id`를 쓰는 테이블을 먼저 만들어 두고, 오늘 실습 구조로 `CREATE TABLE IF NOT EXISTS`를 실행한다.

In [ ]:
# 1과목 챗봇 테이블처럼 conversation_id를 쓰는 테이블을 먼저 만든다
cur.execute("")
cur.execute("""

)""")

# 오늘 실습 구조로 '만들라고' 시킨다
cur.execute("""

""")
print("CREATE 결과:", cur.statusmessage)

# TODO: conn.notices의 마지막 알림(서버가 보낸 NOTICE)을 출력하세요


try:
    cur.execute(""
               )
except Exception as e:
    print("오류 발생:", type(e).__name__, "-", str(e).splitlines()[0])

`CREATE TABLE IF NOT EXISTS`는 실패하지 않는다. 결과도 `CREATE TABLE`로 찍혀서 코드만 보면 새로 만든 것처럼 보인다. 남는 흔적은 서버 알림(NOTICE) 한 줄이다. 실제 구조는 `information_schema`로 확인한다.

In [ ]:
def columns(table):
    # TODO: information_schema.columns에서 public 스키마의 table 테이블 열 이름·자료형을
    #       ordinal_position 순서로 조회하세요 (table은 %s 자리표시자로 넘긴다)

    return cur.fetchall()



| 확인 | 결과 |
| --- | --- |
| `CREATE TABLE IF NOT EXISTS` | 오류 없이 넘어감. 기존 테이블 그대로 |
| 알 수 있는 곳 | 서버 알림(NOTICE), `information_schema.columns` |
| 실제 오류가 나는 곳 | 그다음 INSERT (`session_id` 열 없음) |

> **주의:** 오류가 난 줄(INSERT)과 원인이 있는 줄(CREATE)이 다르다. 열이 없다는 오류가 나면 테이블을 누가 먼저 만들었는지부터 확인한다.

### 2-4. autocommit을 끄면 — 트랜잭션

0절에서 `conn.autocommit = True`로 연결했다. 이 설정을 끈 연결을 하나 더 만들어, 저장한 값이 언제 다른 연결에 보이는지 확인한다.

**트랜잭션(Transaction)이란**: 여러 SQL을 한 묶음으로 처리하는 단위다. `commit()`하면 묶음 전체가 반영되고, `rollback()`하면 묶음 전체가 취소된다.

In [ ]:


# TODO: conn2의 트랜잭션을 확정(commit)하세요


트랜잭션 안에서 오류가 나면 그 뒤 쿼리가 모두 거부된다. 2-2에서 오류 뒤에도 조회가 이어진 것은 autocommit 연결이었기 때문이다.

In [ ]:

# TODO: 실패한 트랜잭션을 취소(rollback)해 conn2를 다시 쓸 수 있게 하세요



| 연결 설정 | 저장한 값이 다른 연결에 보이는 때 | 오류가 난 뒤 |
| --- | --- | --- |
| `autocommit = True` | 실행 즉시 | 다음 쿼리 정상 실행 |
| `autocommit = False` (기본값) | `commit()` 뒤 | `rollback()` 전까지 모든 쿼리 거부 |

> **참고:** 이 실습은 저장 한 번이 한 작업이라 autocommit을 쓴다. "사용자 메시지와 AI 답을 둘 다 저장하거나 둘 다 취소"처럼 묶어야 할 때는 autocommit을 끄고 `commit()`을 직접 부른다.

### 2-5. 세션이 많아지면 — 인덱스

`load()`는 `WHERE session_id = ...`로 찾는다. 행이 적을 때는 차이가 없으므로, 연습용 테이블에 5만 행을 넣고 인덱스 전후 실행 계획을 비교한다.

**인덱스(Index)란**: 특정 열로 빨리 찾도록 따로 만들어 둔 목차다. **실행 계획**은 DB가 쿼리를 어떤 순서로 처리할지 정한 것이고, `EXPLAIN ANALYZE`로 본다.

In [ ]:
#테스트를 위한 테이블 생성 > 5만개 행을 insert
cur.execute("DROP TABLE IF EXISTS demo_index")
cur.execute("CREATE TABLE demo_index (id serial PRIMARY KEY, session_id text, content text)")
# 세션 5,000개 × 10행 = 5만 행을 DB 안에서 만든다
cur.execute("""
INSERT INTO demo_index (session_id, content)
SELECT 'user' || (g % 5000), '메시지 ' || g FROM generate_series(1, 50000) AS g
""")
cur.execute("ANALYZE demo_index")

In [28]:
#쿼리의 실행결과를 분석해 주는 함수 Plan
def plan(sql, params):
    cur.execute("EXPLAIN ANALYZE " + sql, params)
    lines = [r[0].strip() for r in cur.fetchall()]
    scans = [l.lstrip("-> ").split("  (")[0] for l in lines if "Scan" in l]
    return scans, lines[-1]

In [29]:
query = "SELECT content FROM demo_index WHERE session_id = %s ORDER BY id"
print("인덱스 전:", plan(query, ("user42",)))

인덱스 전: (['Seq Scan on demo_index'], 'Execution Time: 3.513 ms')


In [30]:
# TODO: demo_index의 session_id 열에 idx_demo_session 인덱스를 만드세요
# session_id로 인덱스를 생성
cur.execute("CREATE INDEX idx_demo_session ON demo_index (session_id)")

In [35]:
print("인덱스 후:", plan(query, ("user42",)))

인덱스 후: (['Bitmap Heap Scan on demo_index', 'Bitmap Index Scan on idx_demo_session'], 'Execution Time: 0.066 ms')


`Seq Scan`은 5만 행을 처음부터 끝까지 읽은 것이고, `Index Scan`·`Bitmap Index Scan`은 목차로 해당 행만 찾은 것이다. 실행 시간(`Execution Time`)을 비교한다. 오늘 테이블에도 인덱스를 만들어 둔다.

In [36]:
cur.execute("CREATE INDEX idx_session_messages ON session_messages (session_id)")

print("session_messages 인덱스 준비 완료")

session_messages 인덱스 준비 완료


### 2-6. 세션 목록과 삭제

실제 챗봇에는 '내 대화 목록'과 '대화 삭제'가 필요하다. 개인정보 보호를 위해 오래된 대화를 지우는 기능도 만든다.

In [ ]:
def list_session(prefix=""):
    cur.execute("""
        SELECT session_id, count(*), max(created_at)
        FROM session_messages
        WHERE session_id LIKE %s
        GROUP BY session_id
        ORDER BY max(created_at) DESC""", (prefix + "%")
        )
    return cur.fetchall()

def delete_session(session_id):
    cur.execute("DELETE FROM session_messages WHERE session_id = %s", (session_id,))
    return cur.rowcount

def purge_older_than(days):
    cur.execute(
        """
        DELETE FROM session_messages
        WHERE created_at < now() - (%s * interval '1 day')
        """,
        (days,),
    )
    return cur.rowcount



In [ ]:
# userB 대화를 40일 전에 한 것처럼 바꿔 두고, 30일 지난 대화를 지운다


> **주의:** `LIKE %s`에 넘기는 값에도 `%`를 붙여야 "로 시작하는" 검색이 된다(`prefix + "%"`). SQL 문자열 안에 `'user%'`를 직접 쓰면 2-1의 문자열 끼워 넣기 문제로 돌아간다.

### 2-7. 1과목 챗봇 구조와 비교

1과목 챗봇은 사용자(`profiles`), 대화방(`conversations`), 메시지(`messages`)를 나눠 저장했다. 같은 Supabase 프로젝트에 있으므로 구조를 직접 꺼내 비교한다. 1과목 테이블이 없는 프로젝트라면 "없음"으로 나온다.

| 구분 | 1과목 (`conversations` + `messages`) | 오늘 (`session_messages`) |
| --- | --- | --- |
| 대화 구분 | `conversation_id` (대화방 테이블을 참조) | `session_id` 문자열 |
| 사용자 연결 | `conversations.user_id` → `profiles` | 없음 (`session_id`에 사용자 이름을 넣어 흉내) |
| 대화방 제목 | `conversations.title` | 둘 곳이 없음 |
| role 검사 | DB의 CHECK 제약 | 코드 (1-2 `to_messages`) |
| 대화방 삭제 | `on delete cascade`로 메시지도 자동 삭제 | 2-6처럼 직접 삭제 |

오늘 구조는 세션 메모리 원리에 집중하려고 한 테이블에 모았다. 여러 사용자가 쓰는 서비스라면 1과목처럼 나누는 것이 맞다.

### 관찰 정리

- 2-1·2-2·2-3의 세 오류는 각각 어떤 상황에서 실무 서비스에 영향을 줄 수 있을까?
- 2-4에서 autocommit을 끈 연결은 오류 뒤에 무엇을 해야 다시 쓸 수 있었나?
- 2-7에서 1과목 구조와 오늘 구조의 가장 큰 차이는 무엇인가?

## Part 3. Day01(M3) 통합 — 세션 기억을 갖춘 면접 코치

Day01(M3)의 임베딩 지식 검색(`find_related_job`)과 오늘의 세션 저장을 결합해, **이전 대화를 기억하면서 관련 채용 공고까지 스스로 찾아 참고하는** 면접 코치를 만든다.

### 채용 공고 저장소 (Day01 재사용)

In [ ]:

# TODO: job_postings를 embed_documents로 임베딩해 job_vectors에 저장하세요



### find_related_job (Day01 재사용)

In [ ]:

    # TODO: interest를 embed_query로 벡터화하고, job_postings와의 코사인 유사도로 정렬해 상위 k개를 scored에 만드세요



### 방어 로직 (M1 Day04 재사용)

인젝션 방어는 M1 Day04에서 만든 것을 그대로 가져온다.

In [ ]:
# TODO: M1 Day04의 위험 문구·금지어 목록과 input_guard·output_guard를 옮겨오세요

### 세션 기억 + 지식 검색을 갖춘 최종 면접 코치

질문을 저장하고, 관련 채용 공고를 검색하고, 이전 대화까지 참고해 답한다.

In [ ]:
    # TODO: find_related_job으로 message와 관련된 채용 공고 1개를 related_job에 찾고,
    #       이전 대화(load)를 참고하는 SystemMessage + 과거 메시지로 msgs를 만든 뒤
    #       llm.invoke(msgs).content로 answer를 만드세요



### 최종 테스트

In [ ]:

# TODO: 인젝션을 시도하는 질문을 넣어보세요

> **주의:** `find_related_job`은 질문이 어떤 직무와도 실제로 관련 없어도 **가장 가까운 채용 공고 1개를 항상 반환**한다. 처음 버전의 시스템 프롬프트는 이 결과를 '참고 자료'와 '사용자가 실제로 말한 것'을 구분하지 않고 넘겼는데, 그 상태로 `interviewee2`(대화 기록이 비어 있는 세션)에게 검색된 채용 공고 내용을 마치 사용자가 이전에 말한 것처럼 답하는 환각이 실제로 재현됐다. 위 코드는 '대화 기록에 없으면 모른다고 답한다'는 지시를 추가해 이 문제를 막은 버전이다 — M2 Day05(Day09)의 '모르면 정직하게' 원칙이 세션 메모리·지식 검색을 함께 쓸 때도 그대로 필요하다는 뜻이다.

## Part 4. 도구 호출과 세션 메모리 — M2 연결

지금까지는 앱이 과거 대화를 골라 넣었다. Part 4에서는 LLM이 도구를 호출해 필요한 기억을 스스로 꺼내고 저장한다. M2에서 쓴 `@tool`·`bind_tools`·`ToolMessage`·`create_agent`를 그대로 쓴다.

| 절 | 내용 |
| --- | --- |
| 4-1 | 지난 대화를 검색하는 도구 |
| 4-2 | 세션을 넘어 사실을 기억하는 도구 |
| 4-3 (선택) | 도구 호출이 들어간 대화를 DB에 저장 |
| 4-4 (선택) | 잘라 넣을 때 도구 호출과 결과의 짝이 깨지는 오류 |
| 4-5 (선택) | `create_agent`에 세션 메모리 붙이기 |

4-3~4-5는 M4 Day15의 체크포인트 저장기(checkpointer)가 자동으로 해 주는 일을 손으로 짜 보는 선택 과제다. 확장 노트북(투자 리서치 어시스턴트)은 4-5의 `agent_chat`을 쓴다.

### 4-1. 기억을 도구로 — search_history

1-3에서 최근 6개만 넣었을 때 앞의 사실을 잊었다. 이번에는 최근 6개만 넣되, 지난 대화를 검색하는 도구를 준다. 도구는 세션마다 검색 대상이 달라서 함수 안에서 만든다(이유는 도구 정의 아래에서 비교한다).

> **주의:** 도구 안에서는 0절의 `cur`를 쓰지 않고 `with conn.cursor() as c:`로 커서를 새로 연다. 4-5의 `create_agent`는 한 번에 요청된 도구 여러 개를 **동시에** 실행하는데, 커서 하나를 같이 쓰면 한 도구의 조회 결과를 다른 도구가 가져가 버린다. 처음 버전은 `cur`를 같이 썼고, 확장 노트북(`Day11-확장-투자리서치어시스턴트-실습.ipynb`) 테스트 ⑤에서 `list_reports`와 `recall_facts`가 동시에 실행되며 저장된 보고서가 있는데도 둘 다 빈 결과를 돌려줬다.

In [30]:
# 도구 정의
def make_history_tool(session_id):
    #클로저, tools_calls -> name, args
    @tool
    def search_history(keyword: str)-> str:
        """이 사용자가 이전에 나눈 대화에서 keyword가 들어간 메시지를 찾는다. 사용자가 전에 말한 정보를 확인할 때 사용한다."""

        #1. 커서를 생성
        with conn.cursor() as c:
            #2. 세션ID의 모든 메시지 중에서 keyword로 like 검색
            c.execute("""SELECT role, content 
                            FROM session_messages 
                            WHERE session_id = %s AND content ILIKE %s 
                            ORDER BY id""", (session_id, f"%{keyword}%"))
        
            #3. 묶어서 반환
            rows = c.fetchall()
            return "\n".join(f"[{r}] {t}" for r, t in rows) or "관련 대화 없음"

    return search_history

In [31]:
# 도구 llm 에 바인딩
def run_with_tools(msgs, tools, max_steps=5):
    """도구 호출 수행하는 루프함수. 모델이 도구를 부르는 동안 반복해서 수행한다."""
    llm_tools = llm.bind_tools(tools)
    tool_maps = {t.name: t for t in tools}

    for _ in range(max_steps):
        ai = llm_tools.invoke(msgs) #function_calls
        msgs.append(ai)
        if not ai.tool_calls: #툴콜스가 없으면
            return ai.content # 최종 응답
        #툴콜스가 들어왔으면
        for tool in ai.tool_calls: # 툴목록
            print(f"도구 호출 : {tool['name']} args: {tool['args']}")
            tool_result = tool_maps[tool['name']].invoke(tool['args'])
            msgs.append(ToolMessage(str(tool_result), tool_call_id=tool['id']))

    return "[중단]도구호출 실패"


In [34]:
# 테스트 메시지 추가 코드

long_session_id = "long_session_id"


cur.execute(
    "DELETE FROM session_messages WHERE session_id = %s",
    (long_session_id,),
)


test_messages = [
    ("user", "안녕하세요. 제 이름은 이영희입니다."),
    ("assistant", "안녕하세요, 이영희님! 무엇을 도와드릴까요?"),
    ("user", "내가 사는 곳은 서울에 살고 있습니다."),
    ("assistant", "서울에 거주하시는군요."),
    ("user", "요즘 Python과 데이터 분석을 공부하고 있어요."),
    ("assistant", "Python은 데이터 분석에 매우 유용한 언어입니다."),
    ("user", "특히 pandas 사용법에 관심이 많아요."),
    ("assistant", "pandas로 데이터 정제와 집계부터 연습해 보세요."),
]


for role, content in test_messages:
    save(long_session_id, role, content)


In [35]:
msgs = [SystemMessage("최근 대화만 보인다. 사용자가 전에 한 말이 필요하면 search_history 도구로 찾아본다.")]
msgs += to_messages(load_recent("long_session_id", 6)) + [HumanMessage("내 이름이 뭐라고 했지? 사는 곳도 알려 줘.")]

run_with_tools(msgs, [make_history_tool("long_session_id")])

도구 호출 : search_history args: {'keyword': '이름'}
도구 호출 : search_history args: {'keyword': '서울'}


'당신의 이름은 이영희이며, 서울에 살고 있습니다.'

### 4-2. 세션을 넘어 기억하기 — user_facts

세션 메모리는 대화방 안에서만 기억한다. 사용자에 대한 사실을 따로 저장해 두면 새 대화방에서도 꺼낼 수 있다. M4 Day15에서 다룰 장기 기억의 기본 형태다.

### 4-3. (선택) 도구 호출 기록도 저장하기

4-1·4-2의 대화는 DB에 저장하지 않았다. 도구를 쓰는 대화를 저장하려면 role·content만으로는 부족하다. AI 메시지의 `tool_calls`와 도구 결과의 `tool_call_id`까지 저장해야 다시 불러와 이어 갈 수 있다.

### 4-4. (선택) 오류 대응 ④ — 짝이 깨진 기록

1-3처럼 최근 N개만 잘라 넣으면, 도구 결과(`ToolMessage`)만 남고 그 도구를 부른 AI 메시지는 잘려 나갈 수 있다. 확실히 재현하려고 첫 도구 결과부터 잘라서 넣는다.

OpenAI는 도구 결과 앞에 그 결과를 요청한 AI 메시지(`tool_calls`)가 있어야 받는다. 자를 때 도구 결과로 시작하면, 그 결과를 부른 AI 메시지까지 거슬러 올라가 함께 남긴다.

In [ ]:

    
    # TODO: msgs[start]가 ToolMessage인 동안 start를 하나씩 앞으로 옮기세요




### 4-5. (선택) create_agent에 세션 메모리 붙이기

손으로 만든 도구 루프(`run_with_tools`)를 M2 Day08의 `create_agent`로 바꾼다. 세션 기록을 DB에서 불러와 넣고, 에이전트가 이번 턴에 새로 만든 메시지만 다시 저장한다.

In [ ]:

    # TODO: result['messages']에서 불러온 history 뒤에 새로 생긴 메시지만 new_messages로 잘라 내세요


### Day 11(M3 Day02) 정리

| 배운 것 | 핵심 |
| --- | --- |
| 기록과 기억 | 기록은 테이블에 남는 것, 기억은 모델에 보내는 것. 1과목과 같다 |
| SQL 직접 접속 | 자리표시자, 테이블 이름 충돌, 트랜잭션, 인덱스 |
| 도구와 기억 | LLM이 도구로 기억을 찾고(`search_history`) 저장한다(`remember_fact`) |
| 사실 기억 | 세션이 바뀌어도 남는 `user_facts`. M4 Day15 장기 기억으로 이어진다 |

**확인 질문**
- `session_coach`가 답하기 전에 하는 '저장'과 '검색' 두 단계는 각각 무엇을 하는가?
- `search_history`와 `recall_facts`는 각각 어떤 기억을 꺼내는가?

## 확인 문제

1. 1과목 `_ROLE_MAP`과 오늘 `ROLE_TO_MESSAGE`는 각각 무엇을 무엇으로 바꾸는가?
2. 2-1에서 확인한 것처럼, 값을 문자열에 직접 끼워 넣으면 왜 위험한가?
3. 2-3에서 `CREATE TABLE IF NOT EXISTS`가 오류 없이 넘어갔는데도 다음 INSERT가 실패한 이유는?
4. 2-4에서 autocommit을 끈 연결은 오류 뒤에 무엇을 해야 다시 쿼리를 실행할 수 있는가?
5. 1-3에서 최근 6개만 넣었을 때 잃은 것은? 4-1은 그것을 어떻게 되찾았나?
6. 세션 메모리와 `user_facts`의 기억 범위는 어떻게 다른가?
7. (선택) 4-4에서 잘라 낸 기록이 `ToolMessage`로 시작하면 왜 오류가 나는가?